# 03 – So sánh M0–M4, ablation M3, phân tích lỗi

Với mỗi ứng viên, mỗi phương pháp xếp ~10 tin; so thứ tự với nhãn tay. M4 dùng điểm trên **fold kiểm tra** do notebook 02 lưu (`m4_oof.csv`). Kết quả ở đây là kết luận duy nhất về hiệu quả mô hình; dữ liệu ứng tuyển mô phỏng không dùng để kết luận.

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

# Có thể đổi bằng biến môi trường khi chạy thử trên dữ liệu khác
DATA = Path(os.getenv("TH3_DATA_DIR", ROOT / "backend/seed/data"))
LABELS = Path(os.getenv("TH3_LABELS_DIR", ROOT / "backend/seed/labels"))
OUTDIR = Path(os.getenv("TH3_OUT_DIR", ROOT / "docs/report/tables"))
MODELS_OUT = Path(os.getenv("TH3_MODELS_OUT", ROOT / "backend/models_store"))
OUTDIR.mkdir(parents=True, exist_ok=True)


In [2]:
import numpy as np
import pandas as pd
from app.ai import datasets, metrics
from app.ai.embedder import get_embedder

emb = get_embedder()
print("Embedder:", emb.name)  # st:... là model thật; hash-384 chỉ để chạy thử, KHÔNG dùng số liệu để báo cáo
jobs = datasets.load_jobs(DATA / "jobs.csv")
cands = datasets.load_candidates(DATA / "candidates_generated.csv")
datasets.attach_embeddings(jobs, emb)
datasets.attach_embeddings(cands, emb)

LABEL_A = os.getenv("TH3_LABELS_A", "labels_pairs_hung.csv")
LABEL_B = os.getenv("TH3_LABELS_B", "labels_pairs_hoanganh.csv")
raw = datasets.load_labels(LABELS / LABEL_A, LABELS / LABEL_B)
col_a, col_b = [c for c in raw.columns if c.startswith("label_")]
print(f"{len(raw)} cặp có đủ nhãn của cả hai người")


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2140.47it/s]

Embedder: st:paraphrase-multilingual-MiniLM-L12-v2


300 cặp có đủ nhãn của cả hai người


In [3]:
kappa = metrics.weighted_kappa(raw[col_a].to_numpy(), raw[col_b].to_numpy())
print(f"Cohen's kappa (trọng số bình phương) = {kappa:.3f}  (mục tiêu >= 0,6)")

raw["label"] = metrics.merge_labels(raw[col_a].to_numpy(), raw[col_b].to_numpy())
disputed = raw[raw["label"] < 0]
print(f"{len(disputed)} cặp lệch >= 2 bậc cần hai người thảo luận và chốt lại")
disputed.to_csv(OUTDIR / "labels_disputed.csv", index=False)
labels = raw[raw["label"] >= 0].reset_index(drop=True)  # SAU KHI thảo luận: cập nhật file nhãn rồi chạy lại ô này
labels[["candidate_id", "job_id", "label"]].to_csv(OUTDIR / "labels_pairs_final.csv", index=False)
pd.DataFrame([{"kappa_quadratic": kappa, "pairs": len(raw), "disputed": len(disputed)}]).to_csv(OUTDIR / "kappa.csv", index=False)


Cohen's kappa (trọng số bình phương) = 0.959  (mục tiêu >= 0,6)
0 cặp lệch >= 2 bậc cần hai người thảo luận và chốt lại


In [4]:
from app.ai import evaluate

oof = pd.read_csv(OUTDIR / "m4_oof.csv", dtype={"candidate_id": str, "job_id": str})
m4 = labels.merge(oof, on=["candidate_id", "job_id"], how="left")["m4_oof_score"].to_numpy()
assert not np.isnan(m4).any(), "m4_oof.csv không khớp nhãn hiện tại: chạy lại notebook 02"
comparison = evaluate.evaluate_models(labels, cands, jobs, m4_scores=m4, embedder=emb)
comparison.to_csv(OUTDIR / "model_comparison.csv")
comparison.round(4)

,precision@5,ndcg@10,mrr,spearman,roc_auc,n_candidates,ms_per_pair
model,,,,,,,
m0,0.5733,0.9700,0.7417,0.8508,0.9671,30,0.0623
m1,0.5600,0.9622,0.7167,0.8202,0.9019,30,4.9687
m2,0.5467,0.9542,0.7139,0.7848,0.8421,30,0.0728
m3,0.5800,0.9883,0.8000,0.8364,0.9836,30,0.0884
m4,0.5800,0.9704,0.8000,0.8199,0.9809,30,NaN


In [5]:
_, feats = evaluate.build_feature_matrix(labels, cands, jobs, emb)
ablation = evaluate.ablation_m3(labels, feats)
ablation.to_csv(OUTDIR / "ablation_m3.csv", index=False)
ablation.round(4)

,feature_removed,ndcg@10,delta
0,(không bỏ),0.9883,0.0000
1,skill_req,0.9285,-0.0598
2,skill_pref,0.9677,-0.0206
3,semantic,0.9775,-0.0107
4,exp,0.9770,-0.0113
5,edu,0.9886,0.0003
6,pref,0.9886,0.0004


In [6]:
worst = evaluate.error_analysis(labels, cands, jobs, model="m3", n=5, embedder=emb)
worst.to_csv(OUTDIR / "error_analysis_m3.csv", index=False)
worst

,candidate_id,job_id,label,score,error,missing_required,matched_required
0,cand-0082,job-022,3,49,-0.51,SQL|REST API,Node.js
1,cand-0068,job-070,0,50,0.50,Excel|Tư duy phân tích,Giao tiếp
2,cand-0082,job-092,3,50,-0.50,SQL|REST API|OOP,Node.js|Git
3,cand-0068,job-140,0,49,0.49,Excel|Tư duy phân tích|Viết tài liệu,Giao tiếp
4,cand-0285,job-103,0,48,0.48,Python|Excel|Thống kê cơ bản,SQL


## Công bằng

Đổi tên/giới tính ở phần đầu CV, kết quả phải không đổi. Cần hai file CV chỉ khác phần đầu: đặt đường dẫn vào `FAIR_A`, `FAIR_B` (tuỳ chọn).

In [7]:
FAIR_A, FAIR_B = os.getenv("TH3_FAIR_A"), os.getenv("TH3_FAIR_B")
if FAIR_A and FAIR_B:
    job = next(iter(jobs.values()))
    res = evaluate.fairness_check(Path(FAIR_A).read_bytes(), Path(FAIR_B).read_bytes(), Path(FAIR_A).name, job, embedder=emb)
    print(res)
else:
    print("Bỏ qua: chưa cung cấp cặp CV để kiểm tra công bằng.")

{'a': {'skills': ['CSS', 'Giao tiếp', 'JavaScript', 'React', 'Tailwind CSS', 'Tư duy logic'], 'years_exp': 3.0, 'education_level': 3, 'score_m3': 67}, 'b': {'skills': ['CSS', 'Giao tiếp', 'JavaScript', 'React', 'Tailwind CSS', 'Tư duy logic'], 'years_exp': 3.0, 'education_level': 3, 'score_m3': 67}, 'identical': True}
